# kafka2parquet — a step-by-step Kafka → Parquet consumer for telemouse

telemouse's capture agent **produces** three Kafka topics and nothing in the
workspace **consumes** them yet. This notebook builds that consumer one piece
at a time, so you can run each cell, look at the result, and understand why the
next piece is shaped the way it is.

What we end up with:

```
mouse.sessions ─┐
mouse.events  ──┼─► poll → decode JSON envelope → flatten → buffer per session
mouse.markers ─┘                                             │
                                                            ▼ (every N rows / T seconds)
                                        data/events/date=…/session_id=…/part-….parquet
                                        data/sessions/…   data/markers/…
                                                            │
                                                            ▼ commit Kafka offsets
```

Sections:

1. Setup — packages, config read from `telemouse.toml`
2. The wire format — what an envelope looks like, using the demo recording as a fixture
3. QPC → UTC — porting `QpcAnchor` to Python, bit-exact
4. Flattening — one Parquet row per mouse event
5. Arrow schema and the first Parquet file
6. Querying with DuckDB
7. A partitioned, rolling Parquet writer
8. Talking to the broker — metadata, watermarks, consumer groups and `__consumer_offsets`
9. The consumer loop with manual offset commits
10. Verifying the archive — duplicates, gaps, sizes
11. From notebook to service
12. Reconciling gaps from the JSONL recordings — the recording is the source of truth

Everything up to section 8 runs **offline** against `recordings/demo-session.jsonl`,
so you can learn the data side without a broker. Sections 8–10 need the broker
named in `telemouse.toml` to be reachable.

## 1. Setup

Five packages:

| package | why |
|---|---|
| `confluent-kafka` | Kafka client. It wraps librdkafka and ships with **zstd** support, which matters because capture produces with zstd compression. `kafka-python` does not decode zstd reliably. |
| `pyarrow` | Arrow tables in memory and the Parquet writer. |
| `duckdb` | Query a tree of Parquet files with SQL, no server. |
| `orjson` | JSON decoder, several times faster than the stdlib. At 1 kHz of events the stdlib is fine, but a 3-hour backfill is 8 M events and the difference shows. |
| `pandas` | Only so DuckDB results render as tables in the notebook. |

Run the next cell once. It installs into whatever Python this notebook's kernel
uses (the `README.md` next to this notebook shows how to make that a venv).

In [ ]:
%pip install -q confluent-kafka pyarrow duckdb orjson pandas

In [ ]:
import os
import sys
import time
import tomllib
from pathlib import Path

import orjson
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb

print(sys.version)
print("pyarrow", pa.__version__, "| duckdb", duckdb.__version__)

### Config

The broker list lives in `telemouse.toml` under `[kafka]`, so read it from there
rather than duplicating it. Entries may omit the port (the current config does),
in which case Kafka's default 9092 applies.

`DATA_DIR` is where the Parquet tree goes. It sits next to this notebook and is
git-ignored.

In [ ]:
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "telemouse.toml").exists():
            return p
    raise FileNotFoundError("telemouse.toml not found above " + str(start))

REPO = find_repo_root(Path.cwd())
TOML = tomllib.loads((REPO / "telemouse.toml").read_text(encoding="utf-8"))

def with_port(b: str, default=9092) -> str:
    return b if ":" in b else f"{b}:{default}"

BROKERS = ",".join(with_port(b) for b in TOML["kafka"].get("brokers", ["127.0.0.1"]))
TOPICS = ["mouse.sessions", "mouse.events", "mouse.markers"]   # core/src/wire.rs
FIXTURE = REPO / "recordings" / "demo-session.jsonl"
DATA_DIR = REPO / "tools" / "kafka2parquet" / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("repo    :", REPO)
print("brokers :", BROKERS)
print("fixture :", FIXTURE, f"({FIXTURE.stat().st_size/1024:.0f} KB)")
print("data    :", DATA_DIR)

## 2. The wire format

Every transport in telemouse carries the same thing: one JSON **envelope** per
Kafka message, per UDP datagram, per JSONL line. That is why a recording file is
a perfect stand-in for the topic. The envelope is a tagged enum
(`core/src/wire.rs`):

```json
{"type": "session", ...SessionConfig}
{"type": "batch",   ...Batch}
{"type": "marker",  ...Marker}
```

The `type` tag is also which topic the message went to:
`session → mouse.sessions`, `batch → mouse.events`, `marker → mouse.markers`.
The Kafka **key** is always the `session_id`, so one session stays ordered inside
one partition (each topic currently has exactly one partition anyway).

Let's look at one of each.

In [ ]:
def iter_envelopes(path: Path):
    with open(path, "rb") as f:
        for line in f:
            line = line.strip()
            if line:
                yield orjson.loads(line)

first_of = {}
counts = {}
for env in iter_envelopes(FIXTURE):
    t = env["type"]
    counts[t] = counts.get(t, 0) + 1
    first_of.setdefault(t, env)

print("envelopes by type:", counts)

In [ ]:
import copy, pprint

sess = first_of["session"]
pprint.pprint(sess, width=100)

The things a consumer needs from the session envelope:

- `anchor` — a `(qpc, utc_us, qpc_freq)` triple taken at session start. Every event
  timestamp is a raw `QueryPerformanceCounter` tick; the anchor is what maps it to
  wall time. Section 3 ports that math.
- `mouse_cpi` — counts per inch. Raw `dx/dy` stay raw on the wire; centimetres are
  derived *by consumers*, i.e. by us, at query time.
- `games` — per-game sensitivity and yaw coefficients, for degrees-of-turn later.

Now a batch. Events are trimmed with `skip_serializing_if`: a missing `buttons`,
`wheel`, `wheel_h` or `device_ix` **means zero**. Our flattener has to default them.

In [ ]:
b = copy.deepcopy(first_of["batch"])
events = b.pop("events")
pprint.pprint(b, width=100)
print(f"\n{len(events)} events; first three:")
pprint.pprint(events[:3])
print("\nkeys seen across all events in this batch:", sorted({k for e in events for k in e}))

In [ ]:
pprint.pprint(first_of.get("marker", "(no marker in this fixture)"), width=100)

## 3. QPC → UTC, bit-exact

`core/src/clock.rs`:

```rust
pub fn qpc_to_utc_us(&self, qpc: u64) -> i64 {
    let dticks = qpc as i128 - self.qpc as i128;
    let dus = if self.qpc_freq == 10_000_000 { dticks / 10 }
              else { dticks * 1_000_000 / self.qpc_freq as i128 };
    (self.utc_us as i128 + dus) as i64
}
```

The comment in that file promises the result is *exactly reproducible in any
consumer*, so we must match it exactly. The one trap: Rust integer division
**truncates toward zero**, Python's `//` **floors**. They agree for positive
deltas and differ by one for negative ones (an event stamped before the anchor,
which can happen in the first few milliseconds). So we write a truncating divide
and use it.

In [ ]:
from dataclasses import dataclass

def trunc_div(a: int, b: int) -> int:
    q = abs(a) // abs(b)
    return q if (a >= 0) == (b > 0) else -q

assert trunc_div(-7, 10) == 0 and (-7 // 10) == -1     # the difference we are avoiding
assert trunc_div(7, 10) == 0 and trunc_div(-20, 10) == -2

@dataclass(frozen=True)
class QpcAnchor:
    qpc: int
    utc_us: int
    qpc_freq: int

    @classmethod
    def from_session(cls, sess: dict) -> "QpcAnchor":
        a = sess["anchor"]
        return cls(a["qpc"], a["utc_us"], a["qpc_freq"])

    def qpc_to_utc_us(self, qpc: int) -> int:
        dticks = qpc - self.qpc
        if self.qpc_freq == 10_000_000:
            dus = trunc_div(dticks, 10)
        else:
            dus = trunc_div(dticks * 1_000_000, self.qpc_freq)
        return self.utc_us + dus

anchor = QpcAnchor.from_session(sess)
anchor

**Check it against the data.** `Batch.ts_anchor_us` is documented as "UTC µs of the
first event in `events`, mapped via the session anchor" — so for every batch in the
fixture, our port applied to `events[0].ts_qpc` must reproduce `ts_anchor_us`
exactly. If this prints zero mismatches, the port is right.

In [ ]:
mismatches = 0
checked = 0
for env in iter_envelopes(FIXTURE):
    if env["type"] != "batch" or not env["events"]:
        continue
    checked += 1
    got = anchor.qpc_to_utc_us(env["events"][0]["ts_qpc"])
    if got != env["ts_anchor_us"]:
        mismatches += 1
        if mismatches <= 3:
            print("mismatch seq", env["seq_no"], "got", got, "want", env["ts_anchor_us"])
print(f"checked {checked} batches, {mismatches} mismatches")

## 4. Flattening: one row per event

The design choice: **one wide table, one row per mouse event**, with the
batch-level fields (`game`, `pointer_locked`, cursor, drop counters) repeated on
every row of the batch. That sounds wasteful; in Parquet it is nearly free,
because columns are dictionary- and run-length-encoded and a value repeated 25
times in a row costs a few bits. What you get back is a table you can query with
no joins.

We also carry `ts_utc_us`, derived with the anchor when we know the session. If
the consumer starts mid-session and has not seen the session envelope yet, we
write `null` there and keep `ts_anchor_us` on the row so it can be backfilled.

Columnar from the start: instead of building a list of dicts and converting, we
append straight into one Python list per column. That is what Arrow wants, and
it avoids creating millions of tiny dicts during a backfill.

In [ ]:
EVENT_COLUMNS = [
    "session_id", "seq_no", "ts_qpc", "ts_utc_us",
    "dx", "dy", "buttons", "wheel", "wheel_h", "device_ix",
    "game", "pointer_locked", "screen_w", "screen_h", "cursor_x", "cursor_y",
    "drops_since_last", "abs_frames_since_last", "ts_anchor_us",
]

def new_columns() -> dict[str, list]:
    return {c: [] for c in EVENT_COLUMNS}

def append_batch(cols: dict[str, list], b: dict, anchor: QpcAnchor | None) -> int:
    """Append every event of batch `b` to the column lists. Returns rows added."""
    events = b["events"]
    n = len(events)
    if n == 0:
        return 0
    # Batch-level values, repeated n times.
    sid = b["session_id"]
    cols["session_id"].extend([sid] * n)
    cols["seq_no"].extend([b["seq_no"]] * n)
    cols["game"].extend([b.get("game")] * n)
    cols["pointer_locked"].extend([b["pointer_locked"]] * n)
    cols["screen_w"].extend([b["screen_w"]] * n)
    cols["screen_h"].extend([b["screen_h"]] * n)
    cols["cursor_x"].extend([b.get("cursor_x")] * n)
    cols["cursor_y"].extend([b.get("cursor_y")] * n)
    cols["drops_since_last"].extend([b["drops_since_last"]] * n)
    cols["abs_frames_since_last"].extend([b.get("abs_frames_since_last", 0)] * n)
    cols["ts_anchor_us"].extend([b["ts_anchor_us"]] * n)
    # Event-level values. Missing keys mean zero (skip_serializing_if on the Rust side).
    ts = [e["ts_qpc"] for e in events]
    cols["ts_qpc"].extend(ts)
    cols["ts_utc_us"].extend([anchor.qpc_to_utc_us(t) for t in ts] if anchor else [None] * n)
    cols["dx"].extend([e["dx"] for e in events])
    cols["dy"].extend([e["dy"] for e in events])
    cols["buttons"].extend([e.get("buttons", 0) for e in events])
    cols["wheel"].extend([e.get("wheel", 0) for e in events])
    cols["wheel_h"].extend([e.get("wheel_h", 0) for e in events])
    cols["device_ix"].extend([e.get("device_ix", 0) for e in events])
    return n

cols = new_columns()
added = append_batch(cols, first_of["batch"], anchor)
print("rows added:", added)
for c in EVENT_COLUMNS:
    print(f"{c:>22}: {cols[c][:4]}")

## 5. The Arrow schema and the first Parquet file

Parquet is typed, so we say exactly what each column is rather than letting
Arrow guess from the first value (a guess of `int64` for `dx` would work but
waste space; a guess of `null` for `cursor_x` on a pointer-locked session would
break the schema when a desktop session comes along).

Widths mirror the Rust structs: `dx/dy: i32`, `buttons: u16`, `wheel: i16`,
`device_ix: u8`, `seq_no/ts_qpc: u64`, `ts_utc_us: i64`. `session_id` and `game`
are dictionary-encoded strings — that is what makes repeating them per row cheap.

In [ ]:
EVENTS_SCHEMA = pa.schema([
    pa.field("session_id", pa.dictionary(pa.int32(), pa.string()), nullable=False),
    pa.field("seq_no", pa.uint64(), nullable=False),
    pa.field("ts_qpc", pa.uint64(), nullable=False),
    pa.field("ts_utc_us", pa.int64()),                       # null until the session anchor is known
    pa.field("dx", pa.int32(), nullable=False),
    pa.field("dy", pa.int32(), nullable=False),
    pa.field("buttons", pa.uint16(), nullable=False),
    pa.field("wheel", pa.int16(), nullable=False),
    pa.field("wheel_h", pa.int16(), nullable=False),
    pa.field("device_ix", pa.uint8(), nullable=False),
    pa.field("game", pa.dictionary(pa.int32(), pa.string())),
    pa.field("pointer_locked", pa.bool_(), nullable=False),
    pa.field("screen_w", pa.uint32(), nullable=False),
    pa.field("screen_h", pa.uint32(), nullable=False),
    pa.field("cursor_x", pa.int32()),
    pa.field("cursor_y", pa.int32()),
    pa.field("drops_since_last", pa.uint32(), nullable=False),
    pa.field("abs_frames_since_last", pa.uint32(), nullable=False),
    pa.field("ts_anchor_us", pa.int64(), nullable=False),
])

SESSIONS_SCHEMA = pa.schema([
    pa.field("session_id", pa.string(), nullable=False),
    pa.field("started_utc_us", pa.int64(), nullable=False),
    pa.field("qpc_freq", pa.uint64(), nullable=False),
    pa.field("anchor_qpc", pa.uint64(), nullable=False),
    pa.field("anchor_utc_us", pa.int64(), nullable=False),
    pa.field("anchor_uncertainty_us", pa.int64()),
    pa.field("mouse_cpi", pa.float64(), nullable=False),
    pa.field("coalesce_ms", pa.uint64()),
    pa.field("capture_version", pa.string()),
    pa.field("devices", pa.list_(pa.string())),
    pa.field("games_json", pa.string()),      # small nested maps: keep as JSON text, parse in SQL
    pa.field("monitors_json", pa.string()),
])

MARKERS_SCHEMA = pa.schema([
    pa.field("session_id", pa.string(), nullable=False),
    pa.field("seq_no", pa.uint64(), nullable=False),
    pa.field("ts_qpc", pa.uint64(), nullable=False),
    pa.field("ts_utc_us", pa.int64(), nullable=False),
    pa.field("label", pa.string(), nullable=False),
])

def session_row(s: dict) -> dict:
    a = s["anchor"]
    return {
        "session_id": s["session_id"],
        "started_utc_us": s["started_utc_us"],
        "qpc_freq": s["qpc_freq"],
        "anchor_qpc": a["qpc"],
        "anchor_utc_us": a["utc_us"],
        "anchor_uncertainty_us": s.get("anchor_uncertainty_us"),
        "mouse_cpi": s["mouse_cpi"],
        "coalesce_ms": s.get("coalesce_ms"),
        "capture_version": s.get("capture_version"),
        "devices": s.get("devices", []),
        "games_json": orjson.dumps(s.get("games", {})).decode(),
        "monitors_json": orjson.dumps(s.get("monitors", [])).decode(),
    }

print(EVENTS_SCHEMA)

Now the whole fixture into one table and one file. Compare the size to the JSONL
it came from — this is the reason to bother with Parquet at all.

In [ ]:
cols = new_columns()
anchors: dict[str, QpcAnchor] = {}
n_rows = 0
t0 = time.perf_counter()
for env in iter_envelopes(FIXTURE):
    if env["type"] == "session":
        anchors[env["session_id"]] = QpcAnchor.from_session(env)
    elif env["type"] == "batch":
        n_rows += append_batch(cols, env, anchors.get(env["session_id"]))
t_flatten = time.perf_counter() - t0

table = pa.Table.from_pydict(cols, schema=EVENTS_SCHEMA)
scratch = DATA_DIR / "_scratch_demo.parquet"
pq.write_table(table, scratch, compression="zstd")

jsonl_kb = FIXTURE.stat().st_size / 1024
parq_kb = scratch.stat().st_size / 1024
print(f"{n_rows:,} events flattened in {t_flatten*1000:.0f} ms "
      f"({n_rows/t_flatten/1e6:.2f} M events/s)")
print(f"JSONL {jsonl_kb:,.0f} KB  →  Parquet {parq_kb:,.0f} KB  ({jsonl_kb/parq_kb:.1f}× smaller)")
print(f"Arrow table in memory: {table.nbytes/1024:,.0f} KB")

In [ ]:
# Which columns cost what? Parquet keeps per-column statistics in the footer.
meta = pq.read_metadata(scratch)
rg = meta.row_group(0)
rows = [(rg.column(i).path_in_schema, rg.column(i).total_compressed_size, str(rg.column(i).encodings))
        for i in range(rg.num_columns)]
rows.sort(key=lambda r: -r[1])
print(f"{'column':>22} {'bytes':>9}  encodings")
for name, size, enc in rows:
    print(f"{name:>22} {size:>9,}  {enc}")

Read that table bottom-up: the repeated batch-level columns (`game`, `screen_w`,
`pointer_locked`, …) are a few hundred bytes for the whole session. Almost all the
bytes are `ts_qpc`, `ts_utc_us`, `dx`, `dy` — the actual signal. Repeating the batch
fields per row cost nothing measurable.

## 6. Querying with DuckDB

DuckDB reads Parquet directly, including a whole directory tree with a glob, and
it understands `key=value` directory names as partition columns
(`hive_partitioning=true`). No load step.

Three queries that show why raw counts on the wire is the right call: cm and
degrees are one expression away.

In [ ]:
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE VIEW ev AS SELECT * FROM read_parquet('{scratch.as_posix()}')")

con.sql("""
    SELECT session_id, count(*) AS events, count(DISTINCT seq_no) AS batches,
           min(ts_utc_us) AS first_us, max(ts_utc_us) AS last_us,
           (max(ts_utc_us) - min(ts_utc_us)) / 1e6 AS duration_s
    FROM ev GROUP BY session_id
""").show()

In [ ]:
# Events per second and total distance in cm (mouse_cpi from the session envelope).
cpi = sess["mouse_cpi"]
con.sql(f"""
    SELECT ts_utc_us // 1000000 AS second,
           count(*) AS events,
           round(sum(sqrt(dx*dx + dy*dy)) / {cpi} * 2.54, 2) AS path_cm,
           max(drops_since_last) AS max_drops
    FROM ev GROUP BY 1 ORDER BY 1 LIMIT 10
""").show()

In [ ]:
# Degrees turned per batch for a known game: counts * sens * yaw_coeff.
games = sess["games"]
game, gs = next(iter(games.items()))
con.sql(f"""
    SELECT seq_no, game, sum(dx) AS dx_counts,
           round(sum(dx) * {gs['sens']} * {gs['yaw_coeff']}, 2) AS yaw_deg
    FROM ev WHERE game = '{game}' GROUP BY 1, 2 ORDER BY 1 LIMIT 8
""").show()

## 7. A partitioned, rolling Parquet writer

One file per session would mean holding a whole session in memory and losing it
all if the process dies. So the writer **buffers per session** and **rolls a file**
when either limit is hit:

- `max_rows` — bounds memory. 500 k rows ≈ 8 minutes at 1 kHz ≈ 40 MB in Arrow.
- `max_age_s` — bounds how stale the archive can be. 60 s means a query can see
  data at most a minute old.

Layout uses Hive-style partition directories, which DuckDB, Spark, and pandas all
recognise:

```
data/events/date=2026-09-04/session_id=s-20260904-042856-5006/part-<utc_ms>-<first_seq>.parquet
data/sessions/session_id=s-…/session.parquet
data/markers/date=…/session_id=s-…/part-….parquet
```

Two correctness details worth learning:

1. **Write to a temp name, then rename.** `os.replace` is atomic on both Windows
   and Linux. A reader (or a crash) never sees a half-written Parquet file.
2. **The writer reports when it flushed** so the caller can commit Kafka offsets
   *after* the data is durable. That ordering is what makes the pipeline
   at-least-once instead of at-most-once (section 9).

In [ ]:
import datetime as dt

def utc_date(us: int) -> str:
    return dt.datetime.fromtimestamp(us / 1e6, tz=dt.timezone.utc).strftime("%Y-%m-%d")

def atomic_write_table(table: pa.Table, final: Path) -> None:
    final.parent.mkdir(parents=True, exist_ok=True)
    tmp = final.with_suffix(".parquet.tmp")
    pq.write_table(table, tmp, compression="zstd")
    os.replace(tmp, final)

class SessionBuffer:
    __slots__ = ("cols", "rows", "opened_at", "first_seq", "first_anchor_us")
    def __init__(self):
        self.cols = new_columns()
        self.rows = 0
        self.opened_at = time.monotonic()
        self.first_seq = None
        self.first_anchor_us = None

class ParquetSink:
    def __init__(self, root: Path, max_rows: int = 500_000, max_age_s: float = 60.0):
        self.root = Path(root)
        self.max_rows = max_rows
        self.max_age_s = max_age_s
        self.anchors: dict[str, QpcAnchor] = {}
        self.buffers: dict[str, SessionBuffer] = {}
        self.files_written = 0
        self.rows_written = 0
        self.rows_without_anchor = 0

    # -- ingest -----------------------------------------------------------
    def add_session(self, s: dict) -> None:
        sid = s["session_id"]
        self.anchors[sid] = QpcAnchor.from_session(s)
        table = pa.Table.from_pylist([session_row(s)], schema=SESSIONS_SCHEMA)
        atomic_write_table(table, self.root / "sessions" / f"session_id={sid}" / "session.parquet")
        self.files_written += 1

    def add_marker(self, m: dict) -> None:
        sid = m["session_id"]
        table = pa.Table.from_pylist([{k: m[k] for k in MARKERS_SCHEMA.names}], schema=MARKERS_SCHEMA)
        path = (self.root / "markers" / f"date={utc_date(m['ts_utc_us'])}" / f"session_id={sid}"
                / f"marker-{m['seq_no']:08d}.parquet")
        atomic_write_table(table, path)
        self.files_written += 1

    def add_batch(self, b: dict) -> None:
        sid = b["session_id"]
        buf = self.buffers.get(sid)
        if buf is None:
            buf = self.buffers[sid] = SessionBuffer()
        if buf.first_seq is None:
            buf.first_seq = b["seq_no"]
            buf.first_anchor_us = b["ts_anchor_us"]
        anchor = self.anchors.get(sid)
        n = append_batch(buf.cols, b, anchor)
        buf.rows += n
        if anchor is None:
            self.rows_without_anchor += n

    def add_envelope(self, env: dict) -> str:
        """Route by tag. Returns the tag, or 'skipped' for anything unrecognised
        (capture occasionally sends an empty '{}' probe)."""
        t = env.get("type")
        if t == "batch":
            self.add_batch(env)
        elif t == "session":
            self.add_session(env)
        elif t == "marker":
            self.add_marker(env)
        else:
            return "skipped"
        return t

    # -- flushing -----------------------------------------------------------
    def _flush(self, sid: str, buf: SessionBuffer) -> Path:
        table = pa.Table.from_pydict(buf.cols, schema=EVENTS_SCHEMA)
        stamp_ms = buf.first_anchor_us // 1000
        path = (self.root / "events" / f"date={utc_date(buf.first_anchor_us)}" / f"session_id={sid}"
                / f"part-{stamp_ms}-{buf.first_seq:08d}.parquet")
        atomic_write_table(table, path)
        self.files_written += 1
        self.rows_written += buf.rows
        del self.buffers[sid]
        return path

    def flush_due(self, force: bool = False) -> list[Path]:
        """If any buffer is over its row or age limit (or force), flush *every*
        buffer. All-or-nothing on purpose: after a non-empty return nothing is
        left in memory, so the caller can commit Kafka offsets knowing every
        polled message is on disk. Returns the files written."""
        now = time.monotonic()
        due = force or any(
            buf.rows >= self.max_rows or now - buf.opened_at >= self.max_age_s
            for buf in self.buffers.values() if buf.rows
        )
        if not due:
            return []
        return [self._flush(sid, buf) for sid, buf in list(self.buffers.items()) if buf.rows]

    def buffered_rows(self) -> int:
        return sum(b.rows for b in self.buffers.values())

Drive it with the fixture, pretending each JSONL line is a Kafka message. A small
`max_rows` forces several files so you can see the rolling behaviour.

In [ ]:
import shutil
FIXTURE_OUT = DATA_DIR / "fixture"
shutil.rmtree(FIXTURE_OUT, ignore_errors=True)

sink = ParquetSink(FIXTURE_OUT, max_rows=3_000, max_age_s=999)
routed = {}
for env in iter_envelopes(FIXTURE):
    tag = sink.add_envelope(env)
    routed[tag] = routed.get(tag, 0) + 1
    for p in sink.flush_due():
        print("rolled:", p.relative_to(FIXTURE_OUT).as_posix())
for p in sink.flush_due(force=True):
    print("final :", p.relative_to(FIXTURE_OUT).as_posix())

print("\nrouted:", routed)
print(f"files {sink.files_written}, rows {sink.rows_written:,}, rows without anchor {sink.rows_without_anchor}")

In [ ]:
# The whole tree as one table, partition columns included.
glob = (FIXTURE_OUT / "events" / "*" / "*" / "*.parquet").as_posix()
con.sql(f"""
    SELECT date, session_id, count(*) AS events, count(DISTINCT seq_no) AS batches,
           count(DISTINCT filename) AS files
    FROM read_parquet('{glob}', hive_partitioning=true, filename=true)
    GROUP BY 1, 2
""").show()

## 8. Talking to the broker

Everything so far was offline. Now the Kafka side, in the same spirit: look before
consuming.

### 8a. What is on the broker?

The admin client gives us topic metadata. Expect the three telemouse topics with
one partition each (capture creates them with `create_topic(topic, 1, 1, …)`), plus
`__consumer_offsets`, Kafka's internal topic where consumer-group positions live.

In [ ]:
from confluent_kafka import Consumer, TopicPartition, KafkaError, KafkaException
from confluent_kafka.admin import AdminClient

admin = AdminClient({"bootstrap.servers": BROKERS})
md_ = admin.list_topics(timeout=10)
print("broker(s):", [f"{b.host}:{b.port}" for b in md_.brokers.values()])
for name, t in sorted(md_.topics.items()):
    print(f"  {name:<22} partitions={len(t.partitions)}")

### 8b. Watermarks — how much data is there?

Every partition has a **low watermark** (oldest offset still retained; retention
here is one week) and a **high watermark** (the next offset to be written). The
difference is how many messages exist right now. This is also the number a
brand-new consumer with `auto.offset.reset=earliest` will read through.

In [ ]:
probe = Consumer({"bootstrap.servers": BROKERS, "group.id": "kafka2parquet-probe",
                  "enable.auto.commit": False})
for topic in TOPICS:
    tp = TopicPartition(topic, 0)
    lo, hi = probe.get_watermark_offsets(tp, timeout=10)
    print(f"{topic:<16} low={lo:>10,}  high={hi:>10,}  messages={hi-lo:,}")

### 8c. Consumer groups and `__consumer_offsets`

A **consumer group** is a name. When a consumer in group *G* commits, the broker
writes a record to `__consumer_offsets` keyed `(G, topic, partition)` whose value
is the offset. That is the entire mechanism behind "resume where I left off".

- Our real consumer will use the group `kafka2parquet`. Its committed offsets are
  what let the service restart and continue.
- The `console-consumer-NNNNN` groups you may see are left by
  `kafka-console-consumer.sh` peeks; they are harmless.
- The `probe` consumer above never commits, so its group leaves no trace.

List the groups and, for ours, compare committed position to the high watermark —
that difference is **lag**, the number of messages not yet archived.

In [ ]:
fut = admin.list_consumer_groups(request_timeout=10)
groups = fut.result()
print("consumer groups:", [g.group_id for g in groups.valid] or "(none yet)")

GROUP = "kafka2parquet"
c_ = Consumer({"bootstrap.servers": BROKERS, "group.id": GROUP, "enable.auto.commit": False})
committed = c_.committed([TopicPartition(t, 0) for t in TOPICS], timeout=10)
for tp in committed:
    lo, hi = c_.get_watermark_offsets(tp, timeout=10)
    pos = tp.offset if tp.offset >= 0 else None      # -1001 = OFFSET_INVALID: nothing committed yet
    lag = (hi - pos) if pos is not None else hi - lo
    print(f"{tp.topic:<16} committed={pos!s:>10}  high={hi:>10,}  lag={lag:,}")
c_.close()

### 8d. Read a few messages by hand

Before automating, read the session envelopes from the beginning with the probe
group and confirm they decode with the same code we tested on the fixture. Note
`msg.key()` is the session id, exactly as `Envelope::key()` promises.

In [ ]:
probe.assign([TopicPartition("mouse.sessions", 0, 0)])   # explicit partition + offset 0: no group needed
seen = 0
deadline = time.monotonic() + 10
while seen < 5 and time.monotonic() < deadline:
    msg = probe.poll(1.0)
    if msg is None:
        continue
    if msg.error():
        print("error:", msg.error()); break
    env = orjson.loads(msg.value())
    print(f"offset={msg.offset()} key={msg.key().decode()!r} type={env.get('type')} "
          f"cpi={env.get('mouse_cpi')} devices={len(env.get('devices', []))}")
    seen += 1
probe.close()
print(seen, "session envelopes read")

## 9. The consumer loop

The shape:

```
loop:
    msg = poll(1s)
    if msg: decode → sink.add_envelope
    if sink.flush_due():           # something just became durable on disk
        consumer.commit()          # so it is safe to record our position
```

Three settings to understand:

- `enable.auto.commit = False` — with auto-commit, librdkafka commits positions
  on a timer whether or not we have written the data. A crash between commit and
  flush would lose messages. We commit only after a flush, so a crash re-delivers
  the buffered messages instead. That is **at-least-once**; the `(session_id, seq_no)`
  columns let a query deduplicate the rare replay.
- `auto.offset.reset = earliest` — only applies when the group has *no* committed
  offset. First run: archive the whole week of retention. Later runs: resume.
- `group.id = kafka2parquet` — the name under which offsets are stored.

A subtlety worth understanding: a commit records the position of **every** message
polled so far, not just the ones we have written. If the sink flushed one session
and left another buffered, committing would record offsets for rows that are still
only in memory, and a crash right then would lose them. That is why `flush_due`
is all-or-nothing: when any buffer is due, every buffer is written. After a
non-empty flush nothing is in memory, so the commit is always safe. The cost is
that a quiet session sometimes gets a smaller file than `max_rows` would suggest;
section 11 mentions the nightly compaction that tidies that up.

One more: librdkafka raises `_NO_OFFSET` if you commit when nothing new has been
polled since the last commit. So the loop counts messages since the last commit
and only commits when that count is non-zero.

In [ ]:
def run_consumer(sink: ParquetSink, *, group=GROUP, max_seconds=None, max_messages=None,
                 log_every=5.0):
    consumer = Consumer({
        "bootstrap.servers": BROKERS,
        "group.id": group,
        "enable.auto.commit": False,
        "auto.offset.reset": "earliest",
        # Let librdkafka prefetch generously: a backfill is throughput-bound.
        "fetch.max.bytes": 50 * 1024 * 1024,
        "queued.max.messages.kbytes": 256 * 1024,
    })
    consumer.subscribe(TOPICS)
    n_msgs = 0
    uncommitted = 0          # messages polled since the last commit
    counts = {}
    started = time.monotonic()
    last_log = started

    def flush_and_commit(force=False):
        nonlocal uncommitted
        flushed = sink.flush_due(force=force)
        if flushed and uncommitted:
            consumer.commit(asynchronous=False)   # everything polled is now on disk
            uncommitted = 0
        for p in flushed:
            print("  wrote", p.relative_to(sink.root).as_posix())

    try:
        while True:
            if max_seconds is not None and time.monotonic() - started >= max_seconds:
                break
            if max_messages is not None and n_msgs >= max_messages:
                break
            msg = consumer.poll(1.0)
            if msg is not None:
                if msg.error():
                    if msg.error().code() == KafkaError._PARTITION_EOF:
                        continue
                    raise KafkaException(msg.error())
                n_msgs += 1
                uncommitted += 1
                try:
                    env = orjson.loads(msg.value())
                except orjson.JSONDecodeError:
                    counts["undecodable"] = counts.get("undecodable", 0) + 1
                else:
                    tag = sink.add_envelope(env)
                    counts[tag] = counts.get(tag, 0) + 1

            flush_and_commit()

            now = time.monotonic()
            if now - last_log >= log_every:
                last_log = now
                print(f"[{now-started:6.1f}s] msgs={n_msgs:,} buffered_rows={sink.buffered_rows():,} "
                      f"rows_written={sink.rows_written:,} files={sink.files_written}")
    finally:
        # Drain what is buffered, commit if anything new was polled, leave the group cleanly.
        flush_and_commit(force=True)
        if uncommitted:                            # polled messages that produced no rows (probes, etc.)
            consumer.commit(asynchronous=False)
        consumer.close()
        print(f"done: {n_msgs:,} messages {counts}, {sink.rows_written:,} rows in {sink.files_written} files, "
              f"{time.monotonic()-started:.1f}s")

Run it for a bounded time. The first run backfills everything the broker has
retained, so `max_seconds=60` may not finish — that is fine, the offsets are
committed after each flush and the next run picks up where this one stopped.
Re-run the cell in 8c between runs and watch `lag` fall.

In [ ]:
LIVE_OUT = DATA_DIR / "live"
live_sink = ParquetSink(LIVE_OUT, max_rows=500_000, max_age_s=30)
run_consumer(live_sink, max_seconds=60)

## 10. Verifying the archive

Three checks that a consumer of this archive will care about:

1. **Duplicates** — at-least-once means a replay can write the same batch twice
   (two files containing the same `(session_id, seq_no)`). Expect zero after a clean
   run; if you `Ctrl-C` mid-flush you may see a few, and the dedupe query shows how
   to hide them.
2. **Gaps** — `seq_no` is a monotonic per-session counter. A gap means the batch
   never reached Kafka (the capture-side sink drops rather than stall) or was lost
   in transit. This is the only place the Kafka path's loss can be measured.
3. **Size** — compression ratio versus the JSONL recordings.

In [ ]:
live_glob = (LIVE_OUT / "events" / "*" / "*" / "*.parquet").as_posix()
con.execute(f"""
    CREATE OR REPLACE VIEW live AS
    SELECT * FROM read_parquet('{live_glob}', hive_partitioning=true, filename=true)
""")

print("per session:")
con.sql("""
    SELECT session_id, count(*) AS events, count(DISTINCT seq_no) AS batches,
           count(DISTINCT filename) AS files,
           count(*) FILTER (WHERE ts_utc_us IS NULL) AS rows_without_utc,
           sum(drops_since_last) AS ring_drops
    FROM live GROUP BY 1 ORDER BY 1
""").show()

In [ ]:
print("duplicate batches (same session_id+seq_no in more than one file):")
con.sql("""
    SELECT session_id, seq_no, count(DISTINCT filename) AS copies
    FROM live GROUP BY 1, 2 HAVING copies > 1 ORDER BY 1, 2 LIMIT 10
""").show()

print("seq_no gaps per session (batches that never arrived):")
con.sql("""
    WITH s AS (SELECT DISTINCT session_id, seq_no FROM live),
         d AS (SELECT session_id, seq_no,
                      seq_no - lag(seq_no) OVER (PARTITION BY session_id ORDER BY seq_no) AS step
               FROM s)
    SELECT session_id, count(*) FILTER (WHERE step > 1) AS gaps,
           coalesce(sum(step - 1) FILTER (WHERE step > 1), 0) AS batches_missing
    FROM d GROUP BY 1 ORDER BY 1
""").show()

In [ ]:
# A deduplicated view, if you ever need one: keep the first file's copy of each batch.
con.sql("""
    CREATE OR REPLACE VIEW live_dedup AS
    SELECT * EXCLUDE (rn) FROM (
        SELECT *, row_number() OVER (PARTITION BY session_id, seq_no, ts_qpc ORDER BY filename) AS rn
        FROM live
    ) WHERE rn = 1
""")

# Size on disk vs. the matching JSONL recordings, where they exist.
parq_bytes = sum(p.stat().st_size for p in LIVE_OUT.rglob("*.parquet"))
sids = [r[0] for r in con.sql("SELECT DISTINCT session_id FROM live").fetchall()]
jsonl_bytes = sum((REPO / "recordings" / f"{s}.jsonl").stat().st_size
                  for s in sids if (REPO / "recordings" / f"{s}.jsonl").exists())
print(f"parquet on disk: {parq_bytes/1e6:,.1f} MB")
if jsonl_bytes:
    print(f"matching JSONL : {jsonl_bytes/1e6:,.1f} MB  ({jsonl_bytes/parq_bytes:.1f}× larger)")

In [ ]:
# Join events to the sessions table to derive physical units without hard-coding cpi.
sess_glob = (LIVE_OUT / "sessions" / "*" / "*.parquet").as_posix()
con.sql(f"""
    SELECT e.session_id, s.mouse_cpi,
           round(sum(sqrt(e.dx*e.dx + e.dy*e.dy)) / s.mouse_cpi * 2.54 / 100, 2) AS path_m,
           round((max(e.ts_utc_us) - min(e.ts_utc_us)) / 6e7, 1) AS minutes
    FROM live e
    JOIN read_parquet('{sess_glob}', hive_partitioning=true) s USING (session_id)
    GROUP BY 1, 2 ORDER BY 1
""").show()

## 11. From notebook to service

What you have now, in order of appearance, *is* the service. To turn it into one:

1. **Extract** cells 3 (`QpcAnchor`), 4 (`append_batch`), 5 (schemas), 7 (`ParquetSink`)
   and 9 (`run_consumer`) into `kafka2parquet.py`, with a `main()` that reads
   `BROKERS` and `DATA_DIR` from the environment and calls
   `run_consumer(sink)` with no time limit.
2. **Handle SIGTERM** so Docker's stop drains and commits: wrap the loop so a signal
   sets a flag the loop checks, and let the `finally` block do the flush + commit it
   already does.
3. **Run it next to the broker.** The broker in `telemouse.toml` is on another
   machine, and that is where the archive should live. A compose service entry:

   ```yaml
   kafka2parquet:
     build: ./tools/kafka2parquet
     environment:
       BROKERS: kafka:9092          # the broker's service name inside the compose network
       DATA_DIR: /data
     volumes:
       - parquet-data:/data
     depends_on:
       kafka: { condition: service_healthy }
     restart: unless-stopped
   ```

   with a `Dockerfile` of `FROM python:3.12-slim`, `pip install -r requirements.txt`,
   `CMD ["python", "kafka2parquet.py"]`.
4. **Watch it** with the cell in 8c: committed offset versus high watermark is the
   one health number that matters.

Things this notebook deliberately left out, in the order they are likely to matter:

- **Compaction of small files.** A 30-second `max_age_s` on a quiet session yields
  small files. A nightly job that rewrites each `session_id=` directory into one file
  (DuckDB `COPY … TO … (FORMAT PARQUET)`) fixes it.
- **Backfilling `ts_utc_us`** for rows written before their session envelope was
  seen (`rows_without_utc` in section 10). Same nightly job, joining on the
  sessions table and applying the anchor.
- **Schema evolution.** New fields on `Batch` or `RawEvent` land as extra JSON keys
  that `append_batch` ignores. Adding a column means adding it to `EVENT_COLUMNS`,
  `append_batch`, and `EVENTS_SCHEMA`; DuckDB reads old and new files together with
  `union_by_name=true`.
- **The analyze crate.** It reads JSONL today. Reading Parquet from Rust is one
  `parquet` crate away and would make it run over this archive directly.

## 12. Reconciling gaps from the JSONL recordings

Section 10 found real gaps: batches that never reached Kafka. That is by design
on the capture side — its Kafka sink **drops rather than stalls** when the broker
is slow or unreachable, so the live path must never be treated as complete. The
local JSONL recording, on the other hand, is written unconditionally (unless the
ctl panel's "save data" switch or `--no-record` turns it off) and *is* complete.

So the roles are:

| | Kafka path | JSONL recording |
|---|---|---|
| latency | seconds | after the session |
| completeness | best effort | source of truth |
| lives on | the broker box | the capture PC |

Reconciliation fills the archive from the recording, **directly into Parquet**,
not by replaying into the topic. Replaying would put out-of-order data into a
partition that is supposed to be ordered and spend retention on data we already
hold; writing Parquet gains everything and costs nothing.

The step is **idempotent**: it asks DuckDB which `seq_no`s are already archived
for the session, streams the JSONL, and writes only the missing batches, into one
extra file in the same partition directory with `-recon` in its name. Run it every
night over every recording and it does nothing when there is nothing to do. It
also writes the session and marker rows if those never arrived — the case where
the broker was down when a session *started*, which leaves `ts_utc_us` null.

Getting the file across: the viz server already exposes `recordings/` for HTTP
download, or a nightly `robocopy` to a share does the same. Here the recordings
are local, so we read them straight from `recordings/`.

In [ ]:
def archived_seqs(con, root: Path, sid: str) -> set[int]:
    """Every seq_no already in the events tree for this session (any date, any file)."""
    files = list((root / "events").glob(f"*/session_id={sid}/*.parquet"))
    if not files:
        return set()
    glob = (root / "events" / "*" / f"session_id={sid}" / "*.parquet").as_posix()
    return {r[0] for r in con.sql(f"SELECT DISTINCT seq_no FROM read_parquet('{glob}')").fetchall()}

def reconcile_session(con, root: Path, jsonl: Path, max_rows: int = 500_000) -> dict:
    """Write every batch (and session/marker) present in `jsonl` but absent from
    the archive under `root`. Returns a small stats dict. Safe to re-run."""
    root = Path(root)
    sid = jsonl.stem
    have = archived_seqs(con, root, sid)
    stats = {"session_id": sid, "archived_before": len(have), "batches_added": 0,
             "rows_added": 0, "files": [], "session_written": False, "markers_written": 0}
    anchor = None
    cols, rows, first_seq, first_anchor_us = new_columns(), 0, None, None

    def roll():
        nonlocal cols, rows, first_seq, first_anchor_us
        if rows == 0:
            return
        table = pa.Table.from_pydict(cols, schema=EVENTS_SCHEMA)
        path = (root / "events" / f"date={utc_date(first_anchor_us)}" / f"session_id={sid}"
                / f"part-{first_anchor_us // 1000}-{first_seq:08d}-recon.parquet")
        atomic_write_table(table, path)
        stats["files"].append(path)
        stats["rows_added"] += rows
        cols, rows, first_seq, first_anchor_us = new_columns(), 0, None, None

    for env in iter_envelopes(jsonl):
        t = env.get("type")
        if t == "session":
            anchor = QpcAnchor.from_session(env)
            target = root / "sessions" / f"session_id={sid}" / "session.parquet"
            if not target.exists():
                atomic_write_table(pa.Table.from_pylist([session_row(env)], schema=SESSIONS_SCHEMA), target)
                stats["session_written"] = True
        elif t == "batch":
            if env["seq_no"] in have or not env["events"]:
                continue
            if first_seq is None:
                first_seq, first_anchor_us = env["seq_no"], env["ts_anchor_us"]
            rows += append_batch(cols, env, anchor)
            stats["batches_added"] += 1
            if rows >= max_rows:
                roll()
        elif t == "marker":
            target = (root / "markers" / f"date={utc_date(env['ts_utc_us'])}" / f"session_id={sid}"
                      / f"marker-{env['seq_no']:08d}.parquet")
            if not target.exists():
                atomic_write_table(pa.Table.from_pylist([{k: env[k] for k in MARKERS_SCHEMA.names}],
                                                        schema=MARKERS_SCHEMA), target)
                stats["markers_written"] += 1
    roll()
    return stats

Run it over every session that is in the live archive **and** has a recording on
this PC. (Point `targets` at all of `recordings/*.jsonl` instead and the archive
also gains the sessions Kafka never saw at all — check the watermark cell in 8b
against `ls recordings` to see if there are any.)

In [ ]:
archived_sessions = [r[0] for r in con.sql("SELECT DISTINCT session_id FROM live").fetchall()]
targets = [REPO / "recordings" / f"{s}.jsonl" for s in archived_sessions
           if (REPO / "recordings" / f"{s}.jsonl").exists()]
print(f"{len(targets)} of {len(archived_sessions)} archived sessions have a local recording\n")

for jsonl in targets:
    t0 = time.perf_counter()
    st = reconcile_session(con, LIVE_OUT, jsonl)
    print(f"{st['session_id']}: had {st['archived_before']:,} batches, "
          f"added {st['batches_added']:,} batches / {st['rows_added']:,} rows "
          f"in {len(st['files'])} file(s), session_written={st['session_written']}, "
          f"markers_written={st['markers_written']}  [{time.perf_counter()-t0:.1f}s]")
    for p in st["files"]:
        print("   ", p.relative_to(LIVE_OUT).as_posix())

Now the section 10 checks again. The view is re-created so DuckDB picks up the
new files. Gaps should be zero, and duplicates should still be zero — the
reconcile wrote only what was missing.

In [ ]:
con.execute(f"""
    CREATE OR REPLACE VIEW live AS
    SELECT * FROM read_parquet('{live_glob}', hive_partitioning=true, filename=true)
""")

print("seq_no gaps after reconcile:")
con.sql("""
    WITH s AS (SELECT DISTINCT session_id, seq_no FROM live),
         d AS (SELECT session_id, seq_no,
                      seq_no - lag(seq_no) OVER (PARTITION BY session_id ORDER BY seq_no) AS step
               FROM s)
    SELECT session_id, count(*) AS batches,
           count(*) FILTER (WHERE step > 1) AS gaps,
           coalesce(sum(step - 1) FILTER (WHERE step > 1), 0) AS batches_missing
    FROM d GROUP BY 1 ORDER BY 1
""").show()

print("duplicates after reconcile:")
con.sql("""
    SELECT count(*) AS duplicate_batches FROM (
        SELECT session_id, seq_no FROM live GROUP BY 1, 2 HAVING count(DISTINCT filename) > 1)
""").show()

print("files per session, reconcile files marked:")
con.sql("""
    SELECT session_id, count(DISTINCT filename) AS files,
           count(DISTINCT filename) FILTER (WHERE filename LIKE '%-recon.parquet') AS recon_files,
           count(*) FILTER (WHERE filename LIKE '%-recon.parquet') AS recon_rows
    FROM live GROUP BY 1 ORDER BY 1
""").show()

Running the reconcile cell a second time should report `added 0 batches` for every
session — that is the idempotency the nightly job relies on.

Where this leaves the design: the live consumer (section 9) gives you an archive
that is seconds behind and *nearly* complete; the reconcile makes it complete once
the recording is available. The gap query is the single check that both halves are
doing their job. If you ever want the **live** consumers (not the archive) to see
every batch, that is a capture-side change — a local spool that retries undelivered
batches — and a separate decision.

In [ ]:
# Housekeeping: drop the scratch file from section 5. The fixture/ and live/ trees stay.
scratch.unlink(missing_ok=True)
con.close()